# MLOps pipeline

In [1]:
import os
os.chdir(os.path.dirname(os.getcwd())) if os.getcwd().endswith("notebooks") else None
print(os.getcwd())

/mnt/d_drive/AI Projects/graphprobe


In [2]:
R='results/mlops/'
print(open(R+'demo_log.txt').read())

### gate-demo
gate: metric=val_macro_f1 min_improvement=0.005 tie_policy=block floor=0.7
1) first model (floor rule), weak but above floor
  v1: PROMOTED rule=first_model | first model: 0.7350 >= floor 0.7
   approved -> v1
2) better candidate -> promoted
  v2: PROMOTED rule=improvement | +0.0590 >= required 0.005
   approved -> v2
3) regression (undertrained) -> blocked
  v3: BLOCKED  rule=regression | -0.4636 is worse than the approved model
   approved -> v2
4) exact tie (same config again) -> blocked by tie_policy
  v4: BLOCKED  rule=tie | exact tie; tie_policy=block
   approved -> v2
5) missing metric -> blocked
  candidate: BLOCKED  rule=missing_metric | candidate has no usable 'val_macro_f1'
6) rollback after v2 turns out to be problematic in production
   approved before: v2 -> after rollback: v1
### drift-demo

== normal (new random half of same data)
  mean_degree_rel_change                 value=0.07372   thr=0.24107   ok
  isolated_fraction_abs_change           value=0.0288

In [3]:
print(open(R+'failure_handling.md').read())
import json;print(json.dumps(json.load(open(R+'drift_demo.json')),indent=1)[:3000])

| case | expected | observed |
|---|---|---|
| isolated target node | 200: prediction from own features; isolated=true + warning | 200: - OK |
| duplicate target ids | 422: duplicate_target_ids | 422: duplicate_target_ids OK |
| invalid edge endpoint | 422: invalid_edge_endpoint | 422: invalid_edge_endpoint OK |
| negative edge endpoint | 422: invalid_edge_endpoint | 422: invalid_edge_endpoint OK |
| wrong feature dimension | 422: bad_feature_dim | 422: bad_feature_dim OK |
| unknown target id | 404: unknown_target_id | 404: unknown_target_id OK |
| empty target list | 422: empty_targets | 422: empty_targets OK |
| duplicate + self-loop edges | 200: cleaned silently | 200: - OK |
| no approved model / empty registry | 503: model_unavailable | 503: model_unavailable |
| corrupted weights (checksum mismatch) | 503: model_unavailable | 503: model_unavailable |

{
 "thresholds(null: mean+3*std over 100 random half-splits)": {
  "mean_degree_rel_change": {
   "threshold": 0.2410692624488855

Live demo commands: see README.